# تدقيق مجمّع لتعارض الاسم والفئة

يعيد هذا الدفتر حساب مؤشرات التقرير من لقطة محلية. لا يطبع أسماء أو مفاتيح أو عناوين أو وسائل اتصال أو إحداثيات. غيّر متغير البيئة FARQ_TARGETS_PARQUET لمسار اللقطة. يتطلب duckdb.


In [ ]:
import os
from pathlib import Path
import duckdb
TARGETS = Path(os.environ.get("FARQ_TARGETS_PARQUET", "/Users/m4pro/farq-targeting-b2b-release/data/targets.parquet"))
assert TARGETS.is_file(), f"Targets parquet not found: {TARGETS}"
con = duckdb.connect()
p = str(TARGETS).replace("'", "''")
con.execute(f"CREATE VIEW targets AS SELECT * FROM read_parquet('{p}')")


## إجمالي السجلات ومؤشرات الاكتمال


In [ ]:
con.execute("""SELECT COUNT(*) AS rows, COUNT(DISTINCT poi_key) AS distinct_poi_keys, COUNT(*) FILTER (WHERE name_ar IS NULL OR TRIM(name_ar)='') AS missing_name_ar, ROUND(100.0*COUNT(*) FILTER (WHERE name_ar IS NULL OR TRIM(name_ar)='')/NULLIF(COUNT(*),0),2) AS missing_name_ar_pct FROM targets""").df()


## تعارضات الاسم العربي في الرياض
التعريف: المدينة Riyadh، الاسم العربي بعد trim وlower، والفئة category_ar مع fallback إلى category. تُعرض الأعداد فقط.


In [ ]:
con.execute("""WITH x AS (SELECT LOWER(TRIM(name_ar)) nm, COALESCE(NULLIF(TRIM(category_ar),''),NULLIF(TRIM(category),'')) cat, poi_key FROM targets WHERE city_en='Riyadh' AND name_ar IS NOT NULL AND TRIM(name_ar)<>''), g AS (SELECT nm,COUNT(DISTINCT cat) cats FROM x GROUP BY nm) SELECT COUNT(*) name_groups,COUNT(*) FILTER(WHERE cats>1) conflicting_groups,ROUND(100.0*COUNT(*) FILTER(WHERE cats>1)/NULLIF(COUNT(*),0),3) conflicting_pct,(SELECT COUNT(DISTINCT poi_key) FROM x JOIN g USING(nm) WHERE cats>1) affected_keys FROM g""").df()


## فحص أوسع للأسماء بعد تطبيع محدود
يختار الاسم العربي ثم الإنجليزي، ويوحّد بعض الحروف ويحذف المسافات وعلامات الترقيم. المفتاح يتضمن المدينة.


In [ ]:
con.execute("""
WITH b AS (
 SELECT COALESCE(NULLIF(TRIM(name_ar),''),NULLIF(TRIM(name_en),'')) nm0,
        COALESCE(NULLIF(TRIM(city_en),''),NULLIF(TRIM(city_ar),''),'unknown') city,
        COALESCE(NULLIF(TRIM(category_ar),''),NULLIF(TRIM(category),''),'unknown') cat,poi_key
 FROM targets
), n AS (
 SELECT city,cat,poi_key,regexp_replace(translate(lower(nm0),'أإآٱةى','ااااهى'),'[[:space:][:punct:]]+','','g') nm
 FROM b WHERE nm0 IS NOT NULL AND TRIM(nm0)<>''
), g AS (SELECT city,nm,COUNT(DISTINCT cat) cats FROM n GROUP BY city,nm),
metrics AS (
 SELECT 'all cities' scope,COUNT(*) name_city_groups,COUNT(*) FILTER(WHERE cats>1) conflicting_groups,
        ROUND(100.0*COUNT(*) FILTER(WHERE cats>1)/NULLIF(COUNT(*),0),3) conflicting_pct
 FROM g
 UNION ALL
 SELECT 'Riyadh',COUNT(*),COUNT(*) FILTER(WHERE cats>1),
        ROUND(100.0*COUNT(*) FILTER(WHERE cats>1)/NULLIF(COUNT(*),0),3)
 FROM g WHERE city='Riyadh'
)
SELECT m.*,(SELECT COUNT(DISTINCT poi_key) FROM n JOIN g USING(city,nm)
 WHERE g.cats>1 AND (m.scope='all cities' OR g.city='Riyadh')) affected_keys
FROM metrics m ORDER BY scope
""").df()


## مرشحو مراجعة للكلمات الدالة
قائمة الكلمات استكشافية والنتائج مرشحات للمراجعة اليدوية فقط. لا تُطبع صفوف المنشآت.


In [ ]:
food="مطعم|مقهى|كافيه|قهوة|مخبز|حلويات|وجبات|مأكولات|مطاعم|restaurant|cafe|coffee|bakery|food"
con.execute(f"""WITH x AS (SELECT *,COALESCE(NULLIF(TRIM(name_ar),''),NULLIF(TRIM(name_en),'')) nm,COALESCE(NULLIF(TRIM(category_ar),''),NULLIF(TRIM(category),''),'') cat FROM targets), c AS (SELECT *,regexp_matches(lower(COALESCE(nm,'')),'مطعم|restaurant') nf,regexp_matches(lower(cat),'{food}') ff FROM x) SELECT COALESCE(cls_confidence,'(missing)') confidence_bucket,COUNT(*) candidate_records FROM c WHERE nf AND NOT ff GROUP BY 1 ORDER BY 1""").df()


## حدود الاستخدام
شغّل على نسخة محلية مصرح بها. لا تحفظ أي مخرجات صفية. اختلاف النتائج قد ينتج عن تغير اللقطة أو إصدار DuckDB أو تعريف التطبيع. عدد المصادر وثقة المصنف لا يثبتان صحة النشاط.
